In [11]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. Postavke povezivanja (koristimo MySQL prema izvorima [3])
USER = 'root'
PASSWORD = '3k0p13!4'
HOST = 'localhost'
DB_NAME = 'fipu_srp_projekt'

# Kreiranje engine-a
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DB_NAME}")

with engine.connect() as conn:
    # Onemogućite provjeru stranih ključeva kako biste mogli isprazniti tablice
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.execute(text("TRUNCATE TABLE fact_support_tickets;"))
    conn.execute(text("TRUNCATE TABLE dim_vrijeme;"))
    conn.execute(text("TRUNCATE TABLE dim_projekt;"))
    conn.execute(text("TRUNCATE TABLE dim_tehnicar;"))
    conn.execute(text("TRUNCATE TABLE dim_prioritet_status;"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    print("Tablice su ispražnjene i spremne za novi unos.")

df = pd.read_csv('Support_tickets_PROCESSED.csv')
# Pretvorba datuma u datetime objekte
df['issue_created'] = pd.to_datetime(df['issue_created'], format='ISO8601')

# --- 2. PUNJENJE DIMENZIJA ---

# DIM_PROJEKT
dim_projekt = df[['issue_proj']].drop_duplicates()
dim_projekt.columns = ['naziv_projekta']
dim_projekt.to_sql('dim_projekt', engine, if_exists='append', index=False)

# DIM_TEHNIČAR (Kombiniramo reportere i assignee)
reporters = df[['issue_reporter']].rename(columns={'issue_reporter': 'id', 'issue_reporter': 'ime'})
assignees = df[['issue_assignee']].rename(columns={'issue_assignee': 'id', 'issue_assignee': 'ime'})
dim_tehnicar = pd.concat([reporters, assignees]).drop_duplicates()
dim_tehnicar.columns = ['ime_prezime']
dim_tehnicar.to_sql('dim_tehnicar', engine, if_exists='append', index=False)

# DIM_PRIORITET_STATUS
dim_ps = df[['issue_priority', 'issue_status']].drop_duplicates()
dim_ps.columns = ['razina_prioriteta', 'naziv_statusa']
dim_ps.to_sql('dim_prioritet_status', engine, if_exists='append', index=False)

# DIM_VRIJEME (Generiranje iz datuma kreiranja)
dates = pd.to_datetime(df['issue_created']).dt.date.unique()
dim_vrijeme = pd.DataFrame({'vrijeme_key': dates})
dim_vrijeme['dan'] = pd.to_datetime(dim_vrijeme['vrijeme_key']).dt.day
dim_vrijeme['mjesec'] = pd.to_datetime(dim_vrijeme['vrijeme_key']).dt.month
dim_vrijeme['godina'] = pd.to_datetime(dim_vrijeme['vrijeme_key']).dt.year
dim_vrijeme['kvartal'] = pd.to_datetime(dim_vrijeme['vrijeme_key']).dt.quarter
dim_vrijeme['dan_u_tjednu'] = pd.to_datetime(dim_vrijeme['vrijeme_key']).dt.day_name()
dim_vrijeme.to_sql('dim_vrijeme', engine, if_exists='append', index=False)

# --- 3. PUNJENJE TABLICE ČINJENICA (FACT TABLE) ---

# Dohvaćanje generiranih ključeva iz baze za mapiranje
db_projekti = pd.read_sql("SELECT projekt_key, projekt_original_id FROM dim_projekt", engine)
db_tehnicari = pd.read_sql("SELECT tehnicar_key, korisnik_original_id FROM dim_tehnicar", engine)
db_ps = pd.read_sql("SELECT prioritet_status_key, razina_prioriteta, naziv_statusa FROM dim_prioritet_status", engine)

# Mapiranje originalnih podataka na strane ključeve
fact = df.copy()
fact = fact.merge(db_projekti, left_on='issue_proj', right_on='projekt_original_id')
fact = fact.merge(db_tehnicari, left_on='issue_reporter', right_on='korisnik_original_id').rename(columns={'tehnicar_key': 'reporter_key'})
fact = fact.merge(db_tehnicari, left_on='issue_assignee', right_on='korisnik_original_id').rename(columns={'tehnicar_key': 'assignee_key'})
fact = fact.merge(db_ps, left_on=['issue_priority', 'issue_status'], right_on=['razina_prioriteta', 'naziv_statusa'])

# Odabir stupaca za finalnu fact tablicu
fact_final = fact[[
    'id', 'projekt_key', 'reporter_key', 'assignee_key', 
    'prioritet_status_key', 'issue_created'
]]
fact_final = fact_final.rename(columns={'created_at': 'vrijeme_key'})

# Load u bazu
fact_final.to_sql('fact_support_tickets', engine, if_exists='append', index=False)
print("ETL proces uspješno završen.")

Tablice su ispražnjene i spremne za novi unos.
ETL proces uspješno završen.
